## Solution

In [ ]:
import json
from pathlib import Path
from textwrap import dedent

import pandas as pd


WORKSPACE_DIR = Path("/workspace")
VERIFIER_DIR = Path("/logs/verifier")
DATA_PATH = WORKSPACE_DIR / "data" / "customer_journey_test_case.csv"
ENGINE_PATH = WORKSPACE_DIR / "attribution_engine.py"
NOTEBOOK_PATH = WORKSPACE_DIR / "notebook.ipynb"
VARIABLES_PATH = VERIFIER_DIR / "notebook_variables.json"


ENGINE_SOURCE = dedent(
    """
    import pandas as pd


    def _normalize_df(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["timestamp"] = pd.to_datetime(out["timestamp"])
        out = out.drop_duplicates()
        out = out.sort_values(
            ["user_id", "timestamp", "channel", "campaign_name"],
            kind="mergesort"
        ).reset_index(drop=True)
        return out


    def _json_float(x):
        return float(x)


    def attribute_conversions(df, lookback_days=14):
        df = _normalize_df(df)

        conversions = df[(df["is_conversion"] == True) & (df["revenue"] > 0)].copy()
        conversions = conversions.sort_values(
            ["timestamp", "user_id", "channel", "campaign_name"],
            kind="mergesort"
        ).reset_index(drop=True)

        per_conversion = []
        channel_totals = {}

        for _, conv in conversions.iterrows():
            user_id = conv["user_id"]
            conv_ts = conv["timestamp"]
            conv_channel = conv["channel"]
            revenue = float(conv["revenue"])
            lookback_start = conv_ts - pd.Timedelta(days=lookback_days)

            user_rows = df[df["user_id"] == user_id].copy()

            excluded_counts = {
                "conversion_row": 0,
                "same_or_after_conversion": 0,
                "non_click": 0,
                "outside_lookback": 0,
            }
            eligible = []

            for _, row in user_rows.iterrows():
                is_same_row = (
                    row["timestamp"] == conv_ts
                    and row["channel"] == conv["channel"]
                    and row["campaign_name"] == conv["campaign_name"]
                    and bool(row["is_conversion"]) == bool(conv["is_conversion"])
                    and float(row["revenue"]) == float(conv["revenue"])
                    and row["interaction_type"] == conv["interaction_type"]
                )

                if is_same_row:
                    excluded_counts["conversion_row"] += 1
                    continue

                if row["timestamp"] >= conv_ts:
                    excluded_counts["same_or_after_conversion"] += 1
                    continue

                if row["interaction_type"] != "click":
                    excluded_counts["non_click"] += 1
                    continue

                if row["timestamp"] < lookback_start:
                    excluded_counts["outside_lookback"] += 1
                    continue

                eligible.append(row)

            eligible_click_count = len(eligible)

            if eligible_click_count > 0:
                eligible_df = pd.DataFrame(eligible).copy()
                eligible_df = eligible_df.sort_values(
                    ["timestamp", "channel", "campaign_name"],
                    ascending=[False, True, True],
                    kind="mergesort"
                ).reset_index(drop=True)
                winner = eligible_df.iloc[0]
                winning_channel = winner["channel"]
                winning_reason = "last_eligible_click"
            else:
                winning_channel = conv_channel
                winning_reason = "fallback_to_conversion_channel"

            channel_totals[winning_channel] = float(channel_totals.get(winning_channel, 0.0) + revenue)

            per_conversion.append(
                {
                    "conversion_timestamp": conv_ts.isoformat(),
                    "user_id": user_id,
                    "conversion_channel": conv_channel,
                    "revenue": _json_float(revenue),
                    "winning_channel": winning_channel,
                    "winning_reason": winning_reason,
                    "eligible_click_count": int(eligible_click_count),
                    "excluded_counts": {
                        "conversion_row": int(excluded_counts["conversion_row"]),
                        "same_or_after_conversion": int(excluded_counts["same_or_after_conversion"]),
                        "non_click": int(excluded_counts["non_click"]),
                        "outside_lookback": int(excluded_counts["outside_lookback"]),
                    },
                }
            )

        per_conversion = sorted(
            per_conversion,
            key=lambda r: (r["conversion_timestamp"], r["user_id"])
        )
        channel_totals = {
            k: float(channel_totals[k])
            for k in sorted(channel_totals)
        }

        total_revenue = float(sum(float(r["revenue"]) for r in per_conversion))

        return {
            "conversion_count": int(len(per_conversion)),
            "total_revenue": float(total_revenue),
            "channel_totals": channel_totals,
            "per_conversion": per_conversion,
        }
    """
)


def create_notebook(path: Path) -> None:
    nb = {
        "cells": [],
        "metadata": {},
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(nb, f)


def load_engine():
    namespace = {}
    exec(ENGINE_SOURCE, namespace)
    return namespace["attribute_conversions"]


def build_required_scenarios(base_df: pd.DataFrame) -> dict:
    df = base_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    conversions = df[(df["is_conversion"] == True) & (df["revenue"] > 0)].sort_values(
        ["timestamp", "user_id"]
    )
    first_conv = conversions.iloc[0]
    conv_ts = first_conv["timestamp"]
    conv_user = first_conv["user_id"]

    scenarios = {}

    scenarios["base_case"] = df.copy()

    s2 = df.copy()
    email_click_mask = (s2["channel"] == "Email") & (s2["interaction_type"] == "click")
    latest_email_click_idx = s2.loc[email_click_mask, "timestamp"].idxmax()
    s2.loc[latest_email_click_idx, "timestamp"] = conv_ts - pd.Timedelta(days=3)
    scenarios["recent_email_click"] = s2

    s3 = df.copy()
    s3 = pd.concat(
        [
            s3,
            pd.DataFrame(
                [
                    {
                        "user_id": conv_user,
                        "timestamp": conv_ts - pd.Timedelta(hours=2),
                        "channel": "Paid Search",
                        "interaction_type": "click",
                        "campaign_name": "Brand_Rescue",
                        "revenue": 0.0,
                        "is_conversion": False,
                    }
                ]
            ),
        ],
        ignore_index=True,
    )
    scenarios["paid_search_override"] = s3

    s4 = df.copy()
    tie_ts = conv_ts - pd.Timedelta(minutes=90)
    s4 = pd.concat(
        [
            s4,
            pd.DataFrame(
                [
                    {
                        "user_id": conv_user,
                        "timestamp": tie_ts,
                        "channel": "Affiliate",
                        "interaction_type": "click",
                        "campaign_name": "Partner_A",
                        "revenue": 0.0,
                        "is_conversion": False,
                    },
                    {
                        "user_id": conv_user,
                        "timestamp": tie_ts,
                        "channel": "Paid Social",
                        "interaction_type": "click",
                        "campaign_name": "Social_A",
                        "revenue": 0.0,
                        "is_conversion": False,
                    },
                ]
            ),
        ],
        ignore_index=True,
    )
    scenarios["tie_break_same_timestamp"] = s4

    s5 = df.copy()
    s5 = pd.concat(
        [
            s5,
            pd.DataFrame(
                [
                    {
                        "user_id": conv_user,
                        "timestamp": conv_ts + pd.Timedelta(days=20),
                        "channel": "Direct Traffic",
                        "interaction_type": "visit",
                        "campaign_name": "Return_Visit",
                        "revenue": 0.0,
                        "is_conversion": False,
                    },
                    {
                        "user_id": conv_user,
                        "timestamp": conv_ts + pd.Timedelta(days=25),
                        "channel": "Email",
                        "interaction_type": "click",
                        "campaign_name": "Winback_25D",
                        "revenue": 0.0,
                        "is_conversion": False,
                    },
                    {
                        "user_id": conv_user,
                        "timestamp": conv_ts + pd.Timedelta(days=31),
                        "channel": "Display Ad",
                        "interaction_type": "impression",
                        "campaign_name": "Retargeting_Late",
                        "revenue": 0.0,
                        "is_conversion": False,
                    },
                    {
                        "user_id": conv_user,
                        "timestamp": conv_ts + pd.Timedelta(days=32),
                        "channel": "Direct Traffic",
                        "interaction_type": "visit",
                        "campaign_name": "Organic_Return",
                        "revenue": 125.0,
                        "is_conversion": True,
                    },
                ]
            ),
        ],
        ignore_index=True,
    )
    scenarios["second_conversion_extension"] = s5

    return scenarios


def main() -> None:
    WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
    VERIFIER_DIR.mkdir(parents=True, exist_ok=True)

    base_df = pd.read_csv(DATA_PATH)

    with open(ENGINE_PATH, "w", encoding="utf-8") as f:
        f.write(ENGINE_SOURCE)

    attribute_conversions = load_engine()
    scenarios = build_required_scenarios(base_df)

    scenario_results = {
        name: attribute_conversions(df.copy(), lookback_days=14)
        for name, df in scenarios.items()
    }

    with open(VARIABLES_PATH, "w", encoding="utf-8") as f:
        json.dump({"scenario_results": scenario_results}, f, indent=2)

    create_notebook(NOTEBOOK_PATH)


if __name__ == "__main__":
    main()
